# Notebook 3: Retrieval-Augmented Generation (RAG)

## What this notebook does

This notebook answers 643 Austrian tax law questions using RAG:

1. **Load** legal text documents from the corpus
2. **Chunk** each document into overlapping text segments
3. **Embed** all chunks with a multilingual sentence-transformer model
4. **Index** the chunk vectors in a FAISS index for fast similarity search
5. **Retrieve** the top-k most relevant chunks for each question
6. **Generate** an answer using a local language model, given the question + retrieved chunks
7. **Save** `rag_submission.csv`

No external API is used. Everything runs on the T4 GPU.

---

## Design decisions

**Why RAG?**
The language model may not have memorized specific Austrian statutory provisions.
RAG provides the model with the exact legal text at inference time, so it can ground
its answer in the actual law rather than relying solely on pretraining knowledge.

**Embedding model: paraphrase-multilingual-MiniLM-L12-v2**
- Handles German text well (trained on 50+ languages including German)
- Small (118 MB), fast to load, runs on CPU in seconds per batch
- Produces 384-dimensional dense vectors — sufficient for semantic search
- Alternatives like `multilingual-e5-base` give slightly better German retrieval
  but are 3x larger — not worth the tradeoff for a student project

**FAISS index: IndexFlatIP (cosine similarity)**
- Exact nearest-neighbor search on normalized vectors = cosine similarity
- For a few thousand chunks, flat exact search is fast enough (<1s total)
- No need for approximate indexing (IVF, HNSW) at this scale

**Chunk size: 300 words, 50-word overlap**
- 300 words ≈ one paragraph of legal text — enough for a complete legal provision
- Overlap ensures that provisions split across chunk boundaries are still retrievable

**Top-k: 3**
- 3 chunks × ~300 words each = ~900 words of context per question
- Fits within the 768-token generation prompt without truncation
- More chunks would add noise without clearly helping quality

**Generation model: Llama 3.1 8B Instruct (4-bit, same as Notebook 2)**
- The local 8B model is the best model we can run locally on T4
- We cannot use the Groq API here because the project rule allows only one API approach

**Estimated runtime:**
- Embedding + indexing: ~2–3 minutes
- Inference on 643 questions: ~5–8 minutes
- Total: ~8–12 minutes

---

## Before running

1. Upload legal documents as `.txt` files to `/content/data/legal_docs/`
2. Upload `dataset_clean.csv` to `/content/`
3. Set `HF_TOKEN` in Colab Secrets (needed to download the generation model)

## Step 1: Install dependencies

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps peft accelerate bitsandbytes xformers
!pip install -q sentence-transformers faiss-cpu

## Step 2: Load secrets and dataset

In [ ]:
import os
import time
import numpy as np
import pandas as pd

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN") or ""
except Exception:
    HF_TOKEN = os.getenv("HF_TOKEN", "")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN is not set. Add it in Colab Secrets.")

df = pd.read_csv("/content/dataset_clean.csv")
print(f"Loaded {len(df)} test questions.")
print(df[["id", "prompt"]].head(3).to_string())

## Step 3: Load and chunk legal documents

We read all `.txt` files from the legal docs folder.
Each file is split into overlapping word-level chunks.

**Why overlapping chunks?**
Legal provisions often span paragraph breaks. If we split strictly at 300-word boundaries,
a provision that starts at word 290 would be split across two chunks.
A 50-word overlap ensures the full provision appears in at least one chunk.

In [ ]:
LEGAL_DOCS_FOLDER = "/content/data/legal_docs"
CHUNK_SIZE = 300      # words per chunk
CHUNK_OVERLAP = 50    # words shared between consecutive chunks

# --- Check that the folder exists and has .txt files ---
if not os.path.isdir(LEGAL_DOCS_FOLDER):
    raise FileNotFoundError(
        f"Legal docs folder not found: {LEGAL_DOCS_FOLDER}\n"
        "Create the folder and upload your .txt legal documents there."
    )

doc_files = sorted([f for f in os.listdir(LEGAL_DOCS_FOLDER) if f.endswith(".txt")])
if not doc_files:
    raise FileNotFoundError(f"No .txt files found in {LEGAL_DOCS_FOLDER}")

print(f"Found {len(doc_files)} document(s):")
for f in doc_files:
    print(f"  - {f}")

# --- Load and chunk ---
all_chunks = []
chunk_sources = []   # track which file each chunk came from

for filename in doc_files:
    filepath = os.path.join(LEGAL_DOCS_FOLDER, filename)
    with open(filepath, "r", encoding="utf-8") as f:
        words = f.read().split()

    pos = 0
    while pos < len(words):
        chunk_text = " ".join(words[pos : pos + CHUNK_SIZE])
        all_chunks.append(chunk_text)
        chunk_sources.append(filename)
        pos += CHUNK_SIZE - CHUNK_OVERLAP

print(f"\nTotal chunks created: {len(all_chunks)}")
print(f"Average chunk length: {sum(len(c) for c in all_chunks) // len(all_chunks)} characters")
print(f"\nSample chunk (first 200 chars):")
print(all_chunks[0][:200])

## Step 4: Embed all chunks and build the FAISS index

We convert each text chunk into a dense vector using a multilingual sentence-transformer.
All vectors are normalized to unit length so that inner-product similarity = cosine similarity.

The FAISS IndexFlatIP (Inner Product) then allows us to search for the
most similar chunks for any query vector in milliseconds.

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer

# This model is 118 MB and handles German text well.
# It runs on CPU — no GPU needed for embedding at this scale.
print("Loading embedding model...")
embed_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

print(f"Embedding {len(all_chunks)} chunks...")
t0 = time.time()
chunk_vecs = embed_model.encode(
    all_chunks,
    show_progress_bar=True,
    convert_to_numpy=True,
    batch_size=64
)
print(f"Embedding done in {time.time() - t0:.1f}s.")

# Normalize to unit length for cosine similarity
faiss.normalize_L2(chunk_vecs)

# Build the flat inner-product index
dim = chunk_vecs.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(chunk_vecs.astype("float32"))

print(f"FAISS index built: {index.ntotal} vectors, dimension {dim}.")

## Step 5: Retrieve top-k chunks for all questions

We embed all 643 questions at once (fast batch operation) and then query the FAISS index.
This gives us the 3 most relevant text chunks for each question.

In [ ]:
TOP_K = 3

questions = df["prompt"].tolist()

print(f"Embedding {len(questions)} questions...")
q_vecs = embed_model.encode(questions, convert_to_numpy=True, batch_size=64, show_progress_bar=True)
faiss.normalize_L2(q_vecs)

print(f"Searching index for top-{TOP_K} chunks per question...")
distances, indices_rag = index.search(q_vecs.astype("float32"), k=TOP_K)

print(f"Retrieval complete.")
print(f"\nExample retrieval for question 0:")
print(f"  Question: {questions[0][:80]}...")
for rank in range(TOP_K):
    ci = indices_rag[0][rank]
    score = distances[0][rank]
    print(f"  Chunk {rank+1} (score={score:.3f}, source={chunk_sources[ci]}):")
    print(f"    {all_chunks[ci][:100]}...")

## Step 6: Load the generation model

We use Llama 3.1 8B Instruct (same base model as in Notebook 2, without fine-tuning).
The model generates answers conditioned on both the question and the retrieved legal text.

Note: This is the base pretrained model. If you have already run Notebook 2 and saved
the fine-tuned adapter, you could load it here instead for potentially better answers.
For simplicity and independence, this notebook uses the base model.

In [ ]:
from huggingface_hub import login
login(token=HF_TOKEN)

from unsloth import FastLanguageModel

# RAG prompts are longer than plain Q&A prompts (they include 3 retrieved chunks).
# We use max_seq_length=1024 to accommodate the extra context.
MAX_SEQ_LEN = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    token=HF_TOKEN
)

FastLanguageModel.for_inference(model)
print("Generation model loaded and ready for inference.")

## Step 7: Build RAG prompts

For each question, we build a prompt that includes:
1. A system message instructing the model to use the provided legal texts
2. The top-3 retrieved chunks as context
3. The original question

The model is then asked to answer based on the provided context.

In [ ]:
RAG_SYSTEM_MSG = (
    "Du bist ein Experte fuer oesterreichisches Steuerrecht. "
    "Dir werden relevante Auszuege aus oesterreichischen Steuergesetzen "
    "und eine Frage gegeben. "
    "Beantworte die Frage kurz und praezise auf Deutsch, "
    "gestuetzt auf die gegebenen Gesetzestexte."
)

def build_llama_chat(system_msg, user_msg):
    """Build a Llama 3.1 chat-format prompt string (inference only, no assistant turn)."""
    return (
        "<|begin_of_text|>"
        "<|start_header_id|>system<|end_header_id|>\n\n"
        f"{system_msg}<|eot_id|>"
        "<|start_header_id|>user<|end_header_id|>\n\n"
        f"{user_msg}<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\n\n"
    )

rag_prompts = []
for i, q in enumerate(questions):
    # Concatenate top-k retrieved chunks as context
    context_parts = [all_chunks[indices_rag[i][rank]] for rank in range(TOP_K)]
    context = "\n\n---\n\n".join(context_parts)
    user_msg = f"Relevante Gesetzestexte:\n\n{context}\n\nFrage: {q}"
    rag_prompts.append(build_llama_chat(RAG_SYSTEM_MSG, user_msg))

print(f"Built {len(rag_prompts)} RAG prompts.")
print(f"Sample prompt length: {len(rag_prompts[0])} characters")

## Step 8: Run RAG inference

RAG prompts are significantly longer than plain prompts because they include the retrieved
context. We use a smaller batch size (2) to avoid out-of-memory errors on T4.

If you get OOM errors, reduce BATCH_SIZE to 1.

In [ ]:
import torch

BATCH_SIZE = 2         # smaller than fine-tuning because prompts are longer
MAX_NEW_TOKENS = 200   # sufficient for a concise Austrian tax law answer

rag_answers = []
t0 = time.time()

print(f"Running RAG inference on {len(rag_prompts)} questions (batch size {BATCH_SIZE})...")

for i in range(0, len(rag_prompts), BATCH_SIZE):
    batch = rag_prompts[i : i + BATCH_SIZE]

    inputs = tokenizer(
        batch,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LEN
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=True
        )

    input_len = inputs["input_ids"].shape[1]
    for out in outputs:
        answer = tokenizer.decode(out[input_len:], skip_special_tokens=True).strip()
        rag_answers.append(answer)

    done = min(i + BATCH_SIZE, len(rag_prompts))
    if done % 100 == 0 or done == len(rag_prompts):
        elapsed = time.time() - t0
        print(f"  {done}/{len(rag_prompts)} | {elapsed:.0f}s")

print(f"\nRAG inference complete in {time.time() - t0:.0f}s ({(time.time()-t0)/60:.1f} min)")

## Step 9: Save results to CSV

In [ ]:
OUTPUT_CSV = "/content/rag_submission.csv"

submission = pd.DataFrame({
    "id":     df["id"].tolist(),
    "answer": rag_answers
})

submission.to_csv(OUTPUT_CSV, index=False)

print(f"Saved: {OUTPUT_CSV}")
print(f"  Total rows: {len(submission)}")
print()
print("Sample answers:")
for _, row in submission.head(3).iterrows():
    print(f"  ID {row['id']}: {str(row['answer'])[:120]}...")

## Step 10: Verify output

In [ ]:
check = pd.read_csv(OUTPUT_CSV)
print(f"Row count: {len(check)} (expected {len(df)}) — {'OK' if len(check) == len(df) else 'MISMATCH'}")
print(f"Columns:   {list(check.columns)}")

empty = (check["answer"].isna() | (check["answer"].str.strip() == "")).sum()
if empty > 0:
    print(f"WARNING: {empty} empty answers. Check prompt length / truncation.")
else:
    print("No empty answers. File looks good.")